# Practical 10 - Deep Pagination, PIT, Scroll and Export

**Dataset:** 100,000 deterministic synthetic events in `pagination_events`

**Stack:** Elasticsearch 9.2.0, Python `elasticsearch` 9.5.1, kernel `es`

This practical compares every practical way to move through a large result set:

1. First 10 records.
2. Next 10 records.
3. `from` + `size` pagination.
4. The 10,000-hit result-window limitation.
5. `search_after`.
6. `search_after` with a point in time (PIT).
7. The Scroll API.
8. The Python `helpers.scan` helper.
9. A full 100,000-record retrieval.
10. A performance comparison.

The assignment is a standalone streaming CSV exporter.

The index is recreated at the start of the notebook, so every run is repeatable.
All PIT and scroll contexts are closed in `finally` blocks.

## Setup - Build the 100,000-document index

The generator uses a fixed seed. `sequence` is unique and acts as the stable
tie-breaker for deep pagination. The index uses one shard so scroll and scan
batch sizes are easy to reason about in this single-node lab.

The sort order used throughout is:

```json
[{"created_at": "asc"}, {"sequence": "asc"}]
```

In [1]:
import csv
import json
import subprocess
import sys
import time
from pathlib import Path

from elasticsearch import BadRequestError, Elasticsearch, helpers

from generate_data import EXPORT_FIELDS, build_records

HERE = Path.cwd()
assert HERE.name == "Practical_10", f"Run this notebook from Practical_10, not {HERE}"

ARTIFACTS = HERE / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)

ES_URL = "http://127.0.0.1:9200"
INDEX = "pagination_events"
PAGE_SIZE = 1000
TOTAL = 100_000
SORT = [{"created_at": "asc"}, {"sequence": "asc"}]
FIELDS = EXPORT_FIELDS

es = Elasticsearch(ES_URL, request_timeout=60)
info = es.info()
print("Connected to Elasticsearch", info["version"]["number"])
print("cluster:", info["cluster_name"], "| node:", info["name"])
print("index:", INDEX, "| documents:", TOTAL)

Connected to Elasticsearch 9.2.0
cluster: my-elasticsearch-cluster | node: my-node
index: pagination_events | documents: 100000


In [2]:
MAPPING = json.loads((HERE / "mapping.json").read_text(encoding="utf-8"))

if es.indices.exists(index=INDEX):
    es.indices.delete(index=INDEX)
es.indices.create(index=INDEX, **MAPPING)
es.indices.put_settings(index=INDEX, settings={"refresh_interval": "-1"})

load_start = time.perf_counter()
loaded, load_errors = helpers.bulk(
    es,
    (
        {"_index": INDEX, "_id": record["event_id"], "_source": record}
        for record in build_records(count=TOTAL)
    ),
    chunk_size=5000,
    raise_on_error=False,
)
load_seconds = time.perf_counter() - load_start

es.indices.put_settings(index=INDEX, settings={"refresh_interval": "1s"})
es.indices.refresh(index=INDEX)
indexed_count = int(es.count(index=INDEX)["count"])

print(f"Loaded: {loaded:,} documents in {load_seconds:.2f}s")
print(f"Bulk errors: {len(load_errors)}")
print(f"Indexed count: {indexed_count:,}")
if load_errors:
    print(load_errors[:2])

assert loaded == TOTAL
assert load_errors == []
assert indexed_count == TOTAL

Loaded: 100,000 documents in 36.18s
Bulk errors: 0
Indexed count: 100,000


## Task 1 - Retrieve the first 10 records

An explicit sort makes "first" stable. Without a sort, Elasticsearch scores
`match_all` equally and does not promise a useful order.

In [3]:
first_page = es.search(
    index=INDEX,
    size=10,
    sort=SORT,
    source_includes=FIELDS,
)
first_hits = first_page["hits"]["hits"]
first_sequences = [hit["_source"]["sequence"] for hit in first_hits]

print("total matching documents:", f"{first_page['hits']['total']['value']:,}")
print("returned hits:", len(first_hits))
print("sequences:", first_sequences)
print("first document:")
print(json.dumps(first_hits[0]["_source"], indent=2))

assert len(first_hits) == 10
assert first_sequences == list(range(1, 11))

total matching documents: 10,000
returned hits: 10
sequences: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
first document:
{
  "sequence": 1,
  "event_id": "EVT-000001",
  "created_at": "2024-01-01T00:00:00Z",
  "customer_id": "C-0913",
  "region": "North",
  "category": "services",
  "amount": 64296,
  "status": "pending"
}


## Task 2 - Retrieve the next 10 records

Using `from_=10` skips the first ten hits. The page is contiguous and does not
overlap the first page.

In [4]:
second_page = es.search(
    index=INDEX,
    from_=10,
    size=10,
    sort=SORT,
    source_includes=FIELDS,
)
second_hits = second_page["hits"]["hits"]
second_sequences = [hit["_source"]["sequence"] for hit in second_hits]

print("sequences:", second_sequences)
assert len(second_hits) == 10
assert second_sequences == list(range(11, 21))
assert set(first_sequences).isdisjoint(second_sequences)

sequences: [11, 12, 13, 14, 15, 16, 17, 18, 19, 20]


## Task 3 - Implement `from` and `size`

For page-number navigation, `from_` is the offset and `size` is the page size.
Python's client uses `from_` because `from` is a reserved Python keyword.

The pages below are checked for the correct length, offset, and non-overlap.

In [5]:
def from_size_page(offset, size=10):
    return es.search(
        index=INDEX,
        from_=offset,
        size=size,
        sort=SORT,
        source_includes=FIELDS,
        track_total_hits=True,
    )


page_1 = from_size_page(0, 10)
page_2 = from_size_page(10, 10)
page_3 = from_size_page(20, 10)

page_sequences = {
    1: [hit["_source"]["sequence"] for hit in page_1["hits"]["hits"]],
    2: [hit["_source"]["sequence"] for hit in page_2["hits"]["hits"]],
    3: [hit["_source"]["sequence"] for hit in page_3["hits"]["hits"]],
}
print(json.dumps(page_sequences, indent=2))
print("total hits:", f"{page_1['hits']['total']['value']:,}")

assert page_sequences[1] == list(range(1, 11))
assert page_sequences[2] == list(range(11, 21))
assert page_sequences[3] == list(range(21, 31))
assert len(set(page_sequences[1] + page_sequences[2] + page_sequences[3])) == 30

{
  "1": [
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    9,
    10
  ],
  "2": [
    11,
    12,
    13,
    14,
    15,
    16,
    17,
    18,
    19,
    20
  ],
  "3": [
    21,
    22,
    23,
    24,
    25,
    26,
    27,
    28,
    29,
    30
  ]
}
total hits: 100,000


## Task 4 - Investigate the deep-pagination limitation

`index.max_result_window` defaults to **10,000**. A request whose
`from + size` exceeds it fails with HTTP 400 before returning hits.

The last legal window is `from_=9,990, size=10`. The next page,
`from_=10,000, size=10`, requests `10,010` hits and is rejected.

Raising the setting is not a real fix: each shard still has to load and sort
all skipped hits, so memory and latency grow with `from`.

In [6]:
settings = es.indices.get_settings(index=INDEX)[INDEX]["settings"]["index"]
max_result_window = int(settings.get("max_result_window", 10_000))
print("index.max_result_window:", max_result_window)

boundary_page = es.search(
    index=INDEX,
    from_=max_result_window - 10,
    size=10,
    sort=SORT,
    source_includes=FIELDS,
)
print("last legal page sequences:", [hit["_source"]["sequence"] for hit in boundary_page["hits"]["hits"]])
assert len(boundary_page["hits"]["hits"]) == 10

deep_error_text = None
try:
    es.search(
        index=INDEX,
        from_=max_result_window,
        size=10,
        sort=SORT,
        source_includes=FIELDS,
    )
except BadRequestError as exc:
    deep_error_text = str(exc)

print("deep request error:", (deep_error_text or "NO ERROR")[:500])
assert deep_error_text is not None
assert "Result window is too large" in deep_error_text
assert "10000" in deep_error_text

index.max_result_window: 10000
last legal page sequences: [9991, 9992, 9993, 9994, 9995, 9996, 9997, 9998, 9999, 10000]
deep request error: BadRequestError(400, 'search_phase_execution_exception', 'Result window is too large, from + size must be less than or equal to: [10000] but was [10010]. See the scroll api for a more efficient way to request large data sets. This limit can be set by changing the [index.max_result_window] index level setting.', Result window is too large, from + size must be less than or equal to: [10000] but was [10010]. See the scroll api for a more efficient way to request large data sets. This limit can be s


## Task 5 - Implement `search_after`

`search_after` is a cursor rather than an offset. The request must use the same
query and sort on every page, and it must pass the **last hit's sort values** to
the next request.

A unique tie-breaker is required. Here `sequence` is unique, so even documents
created on the same day cannot be skipped or duplicated.

In [7]:
def search_after_page(after=None, size=10):
    kwargs = {
        "index": INDEX,
        "size": size,
        "sort": SORT,
        "source_includes": FIELDS,
        "track_total_hits": False,
    }
    if after is not None:
        kwargs["search_after"] = after
    return es.search(**kwargs)


after_page_1 = search_after_page()
after_hits_1 = after_page_1["hits"]["hits"]
after_sequences_1 = [hit["_source"]["sequence"] for hit in after_hits_1]
cursor = after_hits_1[-1]["sort"]

after_page_2 = search_after_page(after=cursor)
after_hits_2 = after_page_2["hits"]["hits"]
after_sequences_2 = [hit["_source"]["sequence"] for hit in after_hits_2]

print("page 1 sequences:", after_sequences_1)
print("cursor:", cursor)
print("page 2 sequences:", after_sequences_2)

assert after_sequences_1 == list(range(1, 11))
assert after_sequences_2 == list(range(11, 21))
assert set(after_sequences_1).isdisjoint(after_sequences_2)

page 1 sequences: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
cursor: [1704067200000, 10]
page 2 sequences: [11, 12, 13, 14, 15, 16, 17, 18, 19, 20]


## Task 6 - Implement `search_after` with a PIT

A PIT freezes the searchable index state for the life of the cursor. This is the
recommended approach when a deep pagination session must remain consistent while
documents are being indexed or deleted.

The search request uses `pit={"id": ..., "keep_alive": "2m"}` and **omits the
index name**. Elasticsearch adds the `_shard_doc` tie-breaker automatically, so
each hit's sort array includes that extra value.

In [8]:
pit = es.open_point_in_time(index=INDEX, keep_alive="2m")
pit_id = pit["id"]
pit_started = time.perf_counter()
pit_sequences = []
pit_requests = 0
pit_after = None

try:
    for _ in range(10):
        kwargs = {
            "size": 10,
            "sort": SORT,
            "pit": {"id": pit_id, "keep_alive": "2m"},
            "source_includes": FIELDS,
            "track_total_hits": False,
        }
        if pit_after is not None:
            kwargs["search_after"] = pit_after

        pit_response = es.search(**kwargs)
        pit_requests += 1
        pit_id = pit_response.get("pit_id", pit_id)
        pit_hits = pit_response["hits"]["hits"]
        if not pit_hits:
            break

        pit_sequences.extend(hit["_source"]["sequence"] for hit in pit_hits)
        pit_after = pit_hits[-1]["sort"]
finally:
    es.close_point_in_time(id=pit_id)

pit_seconds = time.perf_counter() - pit_started
print("PIT pages retrieved:", pit_requests)
print("documents:", len(pit_sequences))
print("first sequences:", pit_sequences[:10])
print("last sequences:", pit_sequences[-10:])
print("elapsed:", round(pit_seconds, 3), "s")

assert pit_requests == 10
assert len(pit_sequences) == 100
assert len(set(pit_sequences)) == 100
assert pit_sequences == list(range(1, 101))

PIT pages retrieved: 10
documents: 100
first sequences: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
last sequences: [91, 92, 93, 94, 95, 96, 97, 98, 99, 100]
elapsed: 0.623 s


## Task 7 - Implement the Scroll API

Scroll keeps a server-side search context alive. It is stateful and is intended
for one-off bulk processing, not for a user-facing page-number UI.

`sort=["_doc"]` is the fastest scroll order when document order does not matter.
The context must be cleared even if processing fails.

In [9]:
scroll_start = time.perf_counter()
scroll_response = es.search(
    index=INDEX,
    scroll="2m",
    size=PAGE_SIZE,
    sort=["_doc"],
    source_includes=FIELDS,
)
scroll_id = scroll_response.get("_scroll_id")
scroll_sequences = []
scroll_batches = 0

try:
    while True:
        scroll_hits = scroll_response["hits"]["hits"]
        scroll_batches += 1
        if not scroll_hits:
            break

        scroll_sequences.extend(hit["_source"]["sequence"] for hit in scroll_hits)
        scroll_response = es.scroll(scroll_id=scroll_id, scroll="2m")
        scroll_id = scroll_response.get("_scroll_id", scroll_id)
finally:
    if scroll_id:
        es.clear_scroll(scroll_id=scroll_id)

scroll_seconds = time.perf_counter() - scroll_start
scroll_http_calls = scroll_batches + 1
print("documents:", f"{len(scroll_sequences):,}")
print("scroll batches:", scroll_batches, "| HTTP calls including clear:", scroll_http_calls)
print("unique sequences:", len(set(scroll_sequences)), "| elapsed:", round(scroll_seconds, 3), "s")

assert len(scroll_sequences) == TOTAL
assert len(set(scroll_sequences)) == TOTAL

documents: 100,000
scroll batches: 101 | HTTP calls including clear: 102
unique sequences: 100000 | elapsed: 8.038 s


## Task 8 - Use the Python `scan` helper

`helpers.scan` wraps the Scroll API and clears the scroll context automatically.
By default it uses `_doc` order and does not preserve an explicit sort, which is
the fastest mode for ETL.

`size` is a per-shard batch size; this index has one shard.

In [10]:
scan_sample = []
for hit in helpers.scan(
    es,
    index=INDEX,
    query={"query": {"match_all": {}}},
    size=100,
    scroll="1m",
    source_includes=FIELDS,
    request_timeout=60,
):
    scan_sample.append(hit["_source"])
    if len(scan_sample) == 1000:
        break

print("sample documents:", len(scan_sample))
print("sample sequences:", [record["sequence"] for record in scan_sample[:10]])
print("sample unique:", len({record["sequence"] for record in scan_sample}))

assert len(scan_sample) == 1000
assert len({record["sequence"] for record in scan_sample}) == 1000

sample documents: 1000
sample sequences: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
sample unique: 1000


## Task 9 - Retrieve all 100,000 records

The same helper is now allowed to exhaust the index. Only sequence numbers are
kept in the notebook after each hit, which keeps memory bounded while still
proving the full result set was retrieved.

In [11]:
scan_start = time.perf_counter()
scan_sequences = []
for hit in helpers.scan(
    es,
    index=INDEX,
    query={"query": {"match_all": {}}},
    size=PAGE_SIZE,
    scroll="2m",
    source_includes=FIELDS,
    request_timeout=60,
):
    scan_sequences.append(hit["_source"]["sequence"])
scan_seconds = time.perf_counter() - scan_start
scan_http_calls_estimate = (TOTAL + PAGE_SIZE - 1) // PAGE_SIZE + 2

print("retrieved:", f"{len(scan_sequences):,} documents")
print("unique sequences:", f"{len(set(scan_sequences)):,}")
print("min/max sequence:", min(scan_sequences), max(scan_sequences))
print("elapsed:", round(scan_seconds, 3), "s")

assert len(scan_sequences) == TOTAL
assert len(set(scan_sequences)) == TOTAL
assert min(scan_sequences) == 1
assert max(scan_sequences) == TOTAL

retrieved: 100,000 documents
unique sequences: 100,000
min/max sequence: 1 100000
elapsed: 7.894 s


## Task 10 - Compare performance

The same source projection and page size (1,000) are used for every full-scan
method. `track_total_hits=False` removes unrelated counting work.

`from` + `size` is measured only up to the legal 10,000-hit window. The full
100,000 retrieval uses `search_after`, PIT + `search_after`, Scroll, and
`helpers.scan`.

In [12]:
def benchmark_from_size():
    started = time.perf_counter()
    documents = 0
    pages = 0
    for offset in range(0, 10_000, PAGE_SIZE):
        response = es.search(
            index=INDEX,
            from_=offset,
            size=PAGE_SIZE,
            sort=SORT,
            source_includes=FIELDS,
            track_total_hits=False,
        )
        documents += len(response["hits"]["hits"])
        pages += 1
    return {
        "documents": documents,
        "requests": pages,
        "seconds": round(time.perf_counter() - started, 3),
        "scope": "limited to max_result_window",
    }


def benchmark_search_after(use_pit=False):
    started = time.perf_counter()
    pit_id = None
    pit_requests = 0
    documents = 0
    batches = 0
    after = None

    if use_pit:
        pit = es.open_point_in_time(index=INDEX, keep_alive="2m")
        pit_id = pit["id"]
        pit_requests += 1

    try:
        while documents < TOTAL:
            kwargs = {
                "size": PAGE_SIZE,
                "sort": SORT,
                "source_includes": FIELDS,
                "track_total_hits": False,
            }
            if use_pit:
                kwargs["pit"] = {"id": pit_id, "keep_alive": "2m"}
            else:
                kwargs["index"] = INDEX
            if after is not None:
                kwargs["search_after"] = after

            response = es.search(**kwargs)
            hits = response["hits"]["hits"]
            if not hits:
                break

            if use_pit:
                pit_id = response.get("pit_id", pit_id)
                pit_requests += 1

            documents += len(hits)
            batches += 1
            after = hits[-1]["sort"]
    finally:
        if use_pit:
            es.close_point_in_time(id=pit_id)
            pit_requests += 1

    return {
        "documents": documents,
        "requests": pit_requests if use_pit else batches,
        "batches": batches,
        "seconds": round(time.perf_counter() - started, 3),
        "scope": "full index",
    }


from_size_metrics = benchmark_from_size()
search_after_metrics = benchmark_search_after(use_pit=False)
pit_metrics = benchmark_search_after(use_pit=True)

METHODS = [
    {
        "method": "from + size",
        **from_size_metrics,
        "docs_per_second": round(from_size_metrics["documents"] / from_size_metrics["seconds"], 1),
        "note": "Only the first 10,000 hits are legal.",
    },
    {
        "method": "search_after",
        **search_after_metrics,
        "docs_per_second": round(search_after_metrics["documents"] / search_after_metrics["seconds"], 1),
        "note": "Stateless cursor; needs a stable unique sort.",
    },
    {
        "method": "PIT + search_after",
        **pit_metrics,
        "docs_per_second": round(pit_metrics["documents"] / pit_metrics["seconds"], 1),
        "note": "Consistent snapshot; adds PIT open/close overhead.",
    },
    {
        "method": "scroll",
        "documents": len(scroll_sequences),
        "requests": scroll_http_calls,
        "batches": scroll_batches,
        "seconds": round(scroll_seconds, 3),
        "scope": "full index",
        "docs_per_second": round(len(scroll_sequences) / scroll_seconds, 1),
        "note": "Stateful cursor; sort=_doc is fastest.",
    },
    {
        "method": "helpers.scan",
        "documents": len(scan_sequences),
        "requests": scan_http_calls_estimate,
        "batches": (TOTAL + PAGE_SIZE - 1) // PAGE_SIZE,
        "seconds": round(scan_seconds, 3),
        "scope": "full index",
        "docs_per_second": round(len(scan_sequences) / scan_seconds, 1),
        "note": "Scroll wrapper; automatically clears the context.",
    },
]

for method in METHODS:
    print(
        f"{method['method']:22s} "
        f"{method['documents']:>7,} docs  "
        f"{method['requests']:>4} requests  "
        f"{method['seconds']:>7.3f}s  "
        f"{method['docs_per_second']:>9,.1f} docs/s"
    )

assert from_size_metrics["documents"] == 10_000
assert search_after_metrics["documents"] == TOTAL
assert pit_metrics["documents"] == TOTAL
assert len(scroll_sequences) == TOTAL
assert len(scan_sequences) == TOTAL

(ARTIFACTS / "benchmark.json").write_text(
    json.dumps(METHODS, indent=2), encoding="utf-8"
)
print("\nWrote:", ARTIFACTS / "benchmark.json")

from + size             10,000 docs    10 requests    0.769s   13,003.9 docs/s
search_after           100,000 docs   100 requests    7.744s   12,913.2 docs/s
PIT + search_after     100,000 docs   102 requests    7.520s   13,297.9 docs/s
scroll                 100,000 docs   102 requests    8.038s   12,441.3 docs/s
helpers.scan           100,000 docs   102 requests    7.894s   12,668.3 docs/s

Wrote: C:\D-Drive\Elastic Search Practicals\Practical_10\artifacts\benchmark.json


## Assignment - Export 100,000 records to CSV

`export_records.py` is a standalone Python application. It uses
`helpers.scan` and writes one CSV row at a time, so it never materializes the
100,000-document result set in memory.

The notebook runs the application as a subprocess and validates the produced
file and summary.

In [13]:
export_start = time.perf_counter()
export_command = [
    sys.executable,
    str(HERE / "export_records.py"),
    "--index", INDEX,
    "--output", str(ARTIFACTS / "export_100000.csv"),
    "--limit", str(TOTAL),
    "--batch-size", str(PAGE_SIZE),
    "--summary-file", str(ARTIFACTS / "export_summary.json"),
]
export_process = subprocess.run(
    export_command,
    cwd=HERE,
    capture_output=True,
    text=True,
    timeout=300,
)
export_wall_seconds = time.perf_counter() - export_start

print("exit code:", export_process.returncode, "| wall time:", round(export_wall_seconds, 3), "s")
print(export_process.stdout[-1200:])
if export_process.returncode != 0:
    print(export_process.stderr[-1200:])
    raise RuntimeError("export_records.py failed")

export_summary = json.loads((ARTIFACTS / "export_summary.json").read_text(encoding="utf-8"))
export_path = ARTIFACTS / "export_100000.csv"
line_count = sum(1 for _ in export_path.open(encoding="utf-8"))

with export_path.open(encoding="utf-8", newline="") as handle:
    reader = csv.reader(handle)
    header = next(reader)
    first_row = next(reader)

print("CSV lines:", f"{line_count:,}")
print("header:", header)
print("first row:", first_row)
print("summary:", json.dumps(export_summary, indent=2))

assert export_summary["rows"] == TOTAL
assert line_count == TOTAL + 1
assert header == FIELDS
assert export_summary["min_sequence"] == 1
assert export_summary["max_sequence"] == TOTAL
assert export_path.stat().st_size > 0

exit code: 0 | wall time: 13.92 s
{
  "index": "pagination_events",
  "output": "C:\\D-Drive\\Elastic Search Practicals\\Practical_10\\artifacts\\export_100000.csv",
  "requested_limit": 100000,
  "rows": 100000,
  "first_row_sequence": 1,
  "last_row_sequence": 100000,
  "min_sequence": 1,
  "max_sequence": 100000,
  "fields": [
    "sequence",
    "event_id",
    "created_at",
    "customer_id",
    "region",
    "category",
    "amount",
    "status"
  ],
  "elapsed_seconds": 10.986,
  "rows_per_second": 9102.8,
  "output_bytes": 7675821,
  "created_at": "2026-11-19T09:00:54.583534+00:00"
}



CSV lines: 100,001
header: ['sequence', 'event_id', 'created_at', 'customer_id', 'region', 'category', 'amount', 'status']
first row: ['1', 'EVT-000001', '2024-01-01T00:00:00Z', 'C-0913', 'North', 'services', '64296', 'pending']
summary: {
  "index": "pagination_events",
  "output": "C:\\D-Drive\\Elastic Search Practicals\\Practical_10\\artifacts\\export_100000.csv",
  "requested_limit": 100000,
  "rows": 100000,
  "first_row_sequence": 1,
  "last_row_sequence": 100000,
  "min_sequence": 1,
  "max_sequence": 100000,
  "fields": [
    "sequence",
    "event_id",
    "created_at",
    "customer_id",
    "region",
    "category",
    "amount",
    "status"
  ],
  "elapsed_seconds": 10.986,
  "rows_per_second": 9102.8,
  "output_bytes": 7675821,
  "created_at": "2026-11-19T09:00:54.583534+00:00"
}


## What each retrieval method is suitable for

| Method | Best use | Main limitation |
|---|---|---|
| `from` + `size` | Shallow page-number navigation | Cost grows with `from`; default 10,000-hit window |
| `search_after` | Deep feeds and infinite scroll | No random page access; needs a unique sort tie-breaker |
| PIT + `search_after` | Long deep-pagination sessions that need a stable snapshot | PIT must be closed; extra server resources |
| Scroll API | One-off bulk retrieval and ETL | Stateful cursor; not for real-time user requests |
| `helpers.scan` | Python batch jobs and CSV/ETL exports | No guaranteed order unless `preserve_order=True` |

The CSV assignment uses `helpers.scan` because full export is a streaming batch
job. `from` + `size` would fail at 10,000; `search_after` is valid but requires
manual cursor and tie-breaker handling; scroll is also valid but the helper
already manages its lifecycle.

In [14]:
PAGINATION_REPORT = {
    "dataset": {
        "index": INDEX,
        "documents": indexed_count,
        "shards": 1,
        "sort": SORT,
        "load_seconds": round(load_seconds, 3),
    },
    "first_page": first_sequences,
    "second_page": second_sequences,
    "deep_pagination": {
        "max_result_window": max_result_window,
        "boundary_from": max_result_window - 10,
        "rejected_from": max_result_window,
        "error": deep_error_text,
    },
    "pit": {
        "documents": len(pit_sequences),
        "requests": pit_requests,
        "seconds": round(pit_seconds, 3),
    },
    "scroll": {
        "documents": len(scroll_sequences),
        "batches": scroll_batches,
        "http_calls": scroll_http_calls,
        "seconds": round(scroll_seconds, 3),
    },
    "scan_sample": {
        "documents": len(scan_sample),
    },
    "full_scan": {
        "documents": len(scan_sequences),
        "min_sequence": min(scan_sequences),
        "max_sequence": max(scan_sequences),
        "seconds": round(scan_seconds, 3),
    },
    "benchmark": METHODS,
    "assignment": export_summary,
}
(ARTIFACTS / "pagination_report.json").write_text(
    json.dumps(PAGINATION_REPORT, indent=2), encoding="utf-8"
)

print("Final counts:")
print("  all records:", f"{indexed_count:,}")
print("  scroll:", f"{len(scroll_sequences):,}")
print("  scan:", f"{len(scan_sequences):,}")
print("  CSV rows:", f"{export_summary['rows']:,}")
print("Wrote:", ARTIFACTS / "pagination_report.json")
print("All Practical 10 assertions passed.")

Final counts:
  all records: 100,000
  scroll: 100,000
  scan: 100,000
  CSV rows: 100,000
Wrote: C:\D-Drive\Elastic Search Practicals\Practical_10\artifacts\pagination_report.json
All Practical 10 assertions passed.
